In [1]:
# Install required packages
!pip install ultralytics kagglehub opencv-python-headless

import kagglehub
import shutil
import os
import json
import yaml
from pathlib import Path



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [2]:
# Download dataset menggunakan kagglehub
path = kagglehub.dataset_download("linkgish/indonesian-plate-number-from-multi-sources")
print(f"Dataset downloaded to: {path}")

# Investigasi struktur folder
!ls -la {path}

100%|██████████| 1.36G/1.36G [00:21<00:00, 68.6MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/linkgish/indonesian-plate-number-from-multi-sources/versions/3
total 16
drwxr-xr-x 4 root root 4096 Sep  4 09:48 .
drwxr-xr-x 3 root root 4096 Sep  4 09:48 ..
drwxr-xr-x 3 root root 4096 Sep  4 09:48 plate_detection_dataset
drwxr-xr-x 3 root root 4096 Sep  4 09:48 plate_text_dataset


In [3]:
# Definisikan path
RAW_PATH = path  # dari kagglehub
WORK_DIR = "/content/ANPR_YOLO"
IMAGES_DIR = os.path.join(WORK_DIR, "images")
LABELS_DIR = os.path.join(WORK_DIR, "labels")
ANNOTATIONS_DIR = os.path.join(WORK_DIR, "annotations")

# Bersihkan dan buat struktur baru
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(LABELS_DIR, exist_ok=True)
os.makedirs(ANNOTATIONS_DIR, exist_ok=True)

# Cari semua file JSON di dalam nested folder
json_files = []
for root, dirs, files in os.walk(RAW_PATH):
    for file in files:
        if file.endswith('.json'):
            json_files.append(os.path.join(root, file))

print(f"Ditemukan {len(json_files)} file JSON")
for jf in json_files:
    print(f" - {jf}")

# Copy semua JSON ke ANNOTATIONS_DIR
for jf in json_files:
    dest = os.path.join(ANNOTATIONS_DIR, os.path.basename(jf))
    shutil.copy2(jf, dest)

# Copy semua gambar ke IMAGES_DIR (flatten)
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
for root, dirs, files in os.walk(RAW_PATH):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext in image_extensions:
            src = os.path.join(root, file)
            dest = os.path.join(IMAGES_DIR, file)
            # Hindari duplikat nama file
            if os.path.exists(dest):
                base, ext2 = os.path.splitext(file)
                counter = 1
                while os.path.exists(os.path.join(IMAGES_DIR, f"{base}_{counter}{ext2}")):
                    counter += 1
                dest = os.path.join(IMAGES_DIR, f"{base}_{counter}{ext2}")
            shutil.copy2(src, dest)

print(f"Gambar terkopi: {len(os.listdir(IMAGES_DIR))} file")
assert json_files, 'annotations.json tidak ditemukan'
assert os.listdir(IMAGES_DIR), 'Tidak ada gambar yang ditemukan'

Ditemukan 1 file JSON
 - /root/.cache/kagglehub/datasets/linkgish/indonesian-plate-number-from-multi-sources/versions/3/plate_detection_dataset/plate_detection_dataset/annotations/annotations.json
Gambar terkopi: 3246 file


In [4]:
import json
import os
from collections import defaultdict

# --- 1. Kumpulkan semua file JSON ---
json_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.json')]
print("📄 JSON files found:", json_files)

# --- 2. Inisialisasi struktur data ---
image_info = {}
annotations_by_image = defaultdict(list)
category_mapping = {}
category_names = []

# --- 3. Proses semua JSON ---
for json_file in json_files:
    json_path = os.path.join(ANNOTATIONS_DIR, json_file)
    with open(json_path, 'r') as f:
        data = json.load(f)

    # --- 3a. Baca categories ---
    for cat in data.get('categories', []):
        old_id = cat['id']
        name = cat['name']
        if old_id not in category_mapping:
            new_id = len(category_names)
            category_mapping[old_id] = new_id
            category_names.append(name)

    # --- 3b. Baca images (DIPERBAIKI: konversi width/height ke float) ---
    for img in data.get('images', []):
        img_id = img['id']
        file_name = img.get('file_name', f"unknown_{img_id}.jpg")

        # FIX: Pastikan width dan height adalah angka, bukan string
        raw_w = img.get('width', 0)
        raw_h = img.get('height', 0)
        try:
            width = float(raw_w)
            height = float(raw_h)
        except (ValueError, TypeError):
            print(f"Warning: image {img_id} has invalid width/height ({raw_w}, {raw_h}), skipping")
            continue

        if width == 0 or height == 0:
            print(f"Warning: image {img_id} width/height 0, skipping")
            continue

        image_info[img_id] = {
            'file_name': file_name,
            'width': width,
            'height': height
        }

    # --- 3c. Baca annotations ---
    for ann in data.get('annotations', []):
        img_id = ann['image_id']
        if img_id not in image_info:
            continue

        bbox = ann['bbox']
        try:
            x, y, w, h = map(float, bbox)
        except (ValueError, TypeError):
            print(f"⚠️ Error parsing bbox {bbox} for image {img_id}, di-skip")
            continue

        if w <= 0 or h <= 0:
            print(f"⚠️ Bbox width/height <=0 untuk image {img_id}, di-skip")
            continue

        old_cat = ann['category_id']
        if old_cat not in category_mapping:
            new_id = len(category_names)
            category_mapping[old_cat] = new_id
            category_names.append(f"class_{old_cat}")

        annotations_by_image[img_id].append({
            'category_id': category_mapping[old_cat],
            'x': x,
            'y': y,
            'w': w,
            'h': h
        })

print(f"Total images: {len(image_info)}")
print(f"Total annotations: {sum(len(v) for v in annotations_by_image.values())}")
print(f"Classes: {category_names}")

# --- 4. Tulis file label .txt untuk setiap gambar ---
for img_id, info in image_info.items():
    img_filename = info['file_name']
    base = os.path.splitext(img_filename)[0]
    label_path = os.path.join(LABELS_DIR, base + '.txt')

    # Ambil width/height sebagai float
    width = info['width']
    height = info['height']

    anns = annotations_by_image.get(img_id, [])
    if not anns:
        Path(label_path).touch()
        continue

    lines = []
    for ann in anns:
        # Potong bbox ke batas image sebelum normalisasi.
        x1 = max(0.0, min(width, ann['x']))
        y1 = max(0.0, min(height, ann['y']))
        x2 = max(0.0, min(width, ann['x'] + ann['w']))
        y2 = max(0.0, min(height, ann['y'] + ann['h']))
        if x2 <= x1 or y2 <= y1:
            continue
        x_center = ((x1 + x2) / 2) / width
        y_center = ((y1 + y2) / 2) / height
        w_norm = (x2 - x1) / width
        h_norm = (y2 - y1) / height

        line = f"{ann['category_id']} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
        lines.append(line)

    with open(label_path, 'w') as f:
        f.write('\n'.join(lines))

print("✅ Konversi selesai! File label telah dibuat.")

📄 JSON files found: ['annotations.json']
Total images: 1377
Total annotations: 2128
Classes: ['plate_number']
✅ Konversi selesai! File label telah dibuat.


In [5]:
from sklearn.model_selection import train_test_split

# Dapatkan semua file gambar
image_files = [f for f in os.listdir(IMAGES_DIR)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))
               and os.path.exists(os.path.join(LABELS_DIR, os.path.splitext(f)[0] + '.txt'))]
assert image_files, 'Tidak ada gambar berlabel yang bisa di-split'

# Split 80:20
train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

# Buat folder train/val
for split in ['train', 'val']:
    os.makedirs(os.path.join(IMAGES_DIR, split), exist_ok=True)
    os.makedirs(os.path.join(LABELS_DIR, split), exist_ok=True)

# Fungsi untuk memindahkan file
def move_files(file_list, split):
    for f in file_list:
        # Pindahkan gambar
        src_img = os.path.join(IMAGES_DIR, f)
        dst_img = os.path.join(IMAGES_DIR, split, f)
        shutil.move(src_img, dst_img)

        # Pindahkan label (jika ada)
        label_name = os.path.splitext(f)[0] + '.txt'
        src_label = os.path.join(LABELS_DIR, label_name)
        dst_label = os.path.join(LABELS_DIR, split, label_name)
        if os.path.exists(src_label):
            shutil.move(src_label, dst_label)

move_files(train_files, 'train')
move_files(val_files, 'val')

print(f"Train: {len(train_files)} images")
print(f"Val: {len(val_files)} images")

Train: 2596 images
Val: 650 images


In [ ]:
# Fix class ID mismatch: reuse category_names dari cell konversi (single source of truth)
# ponytail: satu mapping dipakai untuk label .txt dan data.yaml
if 'category_names' in globals() and category_names:
    class_names = category_names
else:
    # fallback bila notebook di-run out-of-order tanpa cell konversi
    class_names = []
    for json_file in os.listdir(ANNOTATIONS_DIR):
        if json_file.endswith('.json'):
            with open(os.path.join(ANNOTATIONS_DIR, json_file), 'r') as f:
                data = json.load(f)
                for category in data.get('categories', []):
                    name = category['name']
                    if name not in class_names:
                        class_names.append(name)

print(f"Classes found: {class_names}")

# Buat data.yaml
data_yaml = {
    'path': WORK_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(class_names),
    'names': class_names
}

yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml created at: {yaml_path}")
print(yaml.dump(data_yaml, default_flow_style=False))


In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load pretrained YOLOv8n (nano - ringan, cepat)
model = YOLO('yolov8n.pt')

# Train — improved: epochs 100, imgsz 960 untuk small object, seed & cache
results = model.train(
    data=os.path.join(WORK_DIR, 'data.yaml'),
    epochs=100,
    imgsz=960,
    batch=8,
    device=0,
    workers=2,
    patience=20,
    seed=42,
    cache=False,
    deterministic=True,
    project='/content/ANPR_YOLO/runs',
    name='plate_detector',
    exist_ok=True
)

print("✅ Training selesai!")
print(f"Best model: {Path(results.save_dir) / 'weights' / 'best.pt'}")
print(f"Results dir: {results.save_dir}")


In [ ]:
from pathlib import Path
from ultralytics import YOLO

# Robust path via results.save_dir (tidak hardcode / tidak tebak folder)
best_weights = Path(results.save_dir) / "weights" / "best.pt"
assert best_weights.exists(), f'best.pt tidak ditemukan: {best_weights}'

best_model = YOLO(str(best_weights))

# Evaluasi di validation set
metrics = best_model.val(
    data=str(yaml_path), split='val', imgsz=960, conf=0.25,
    plots=True, device=0, verbose=False
)

print("\n📊 Evaluation Results:")
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print(f"Weights: {best_weights}")
print(f"Plots: {Path(best_weights).parent.parent}")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

CONF = 0.25  # ponytail: threshold tunggal untuk evaluasi & inference

# 1. Visualize Training History (path robust via results.save_dir)
try:
    results_csv = Path(results.save_dir) / "results.csv"
except NameError:
    results_csv = Path("runs/detect/ANPR_YOLO/plate_detector/results.csv")

if not results_csv.exists():
    # fallback legacy path Colab
    alt = Path("runs/detect/ANPR_YOLO/plate_detector/results.csv")
    if alt.exists():
        results_csv = alt

if results_csv.exists():
    data = pd.read_csv(results_csv)
    data.columns = data.columns.str.strip()
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for col in ['train/box_loss', 'train/cls_loss', 'train/dfl_loss']:
        if col in data.columns:
            ax[0].plot(data[col], label=col)
    ax[0].set_title("Training Loss")
    ax[0].legend()
    ax[0].grid(alpha=0.3)
    for col in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
        if col in data.columns:
            ax[1].plot(data[col], label=col)
    ax[1].set_title("Validation mAP")
    ax[1].legend()
    ax[1].grid(alpha=0.3)
    plt.show()
else:
    print(f"results.csv tidak ditemukan di {results_csv}")

# 2. Proper detection metrics — menggantikan BCE/MSE manual
# YOLOv8 pakai BCE/varifocal + CIoU + DFL; BCE vs MSE per-sample bukan loss asli
# Gunakan metric deteksi yang sudah dihasilkan Ultralytics
print("\n📊 Metrics (dari best_model.val()):")
print(f"mAP50: {metrics.box.map50:.4f} | mAP50-95: {metrics.box.map:.4f} | P: {metrics.box.mp:.4f} | R: {metrics.box.mr:.4f}")
try:
    print(f"Confusion matrix / PR / F1 curve tersimpan di: {Path(results.save_dir)}")
    print(f"  - {Path(results.save_dir) / 'confusion_matrix.png'}")
    print(f"  - {Path(results.save_dir) / 'PR_curve.png'}")
    print(f"  - {Path(results.save_dir) / 'F1_curve.png'}")
except NameError:
    pass
print(f"conf={CONF} diseragamkan untuk evaluasi & inference")


In [ ]:
import cv2
import matplotlib.pyplot as plt

CONF = 0.25  # ponytail: single threshold konsisten dengan cell evaluasi

# Ambil sample dari validation set
val_images = os.listdir(os.path.join(IMAGES_DIR, 'val'))[:5]

fig, axes = plt.subplots(1, len(val_images), figsize=(15, 5))
for i, img_name in enumerate(val_images):
    img_path = os.path.join(IMAGES_DIR, 'val', img_name)
    pred = best_model(img_path, conf=CONF, verbose=False)

    # Plot hasil
    annotated = pred[0].plot()
    axes[i].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[i].axis('off')
    axes[i].set_title(img_name[:15])

plt.tight_layout()
plt.show()


In [ ]:
# Export ke format ONNX (lebih cepat, cross-platform) — imgsz samakan dengan training
best_model.export(format='onnx', imgsz=960)
